# Vet Clinics in Berlin – Cleaning and Normalization (v1)

This notebook takes the intermediate **v0 vet clinics dataset** produced in:

- `01_vet_clinics_osm_lor_join.ipynb`

and performs cleaning and normalization steps to produce a **v1 dataset**
that is closer to the final schema required for the vet clinics layer.

Key goals:

- Standardize clinic naming conventions.
- Normalize and retain address fields in a structured form.
- Derive basic service flags (e.g. emergency) where possible.
- Extract and structure opening hours and operating days (heuristically).
- Aggregate contact information while keeping structured contact fields.
- Preserve district and neighborhood information (names and IDs).
- Preserve geometry for later use in the unified POI table.
- Keep provenance and source identifiers (`data_source`, `source_osm_id`).

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 80)

# Snapshot date used for provenance metadata. Adjust if needed.
SNAPSHOT_DATE = "2025-12-09"

## 1. Load v0 vet clinics dataset

We read the v0 file produced by the OSM + LOR spatial join:

- `cache/vets_osm_berlin_with_lor_latest_v0.csv`

This dataset includes:

- OSM-derived attributes: `name`, `addr:*`, contact fields, `opening_hours`,
  `operator`, `emergency`, `wheelchair`, etc.
- Location coordinates: `lat`, `lon`.
- LOR-based spatial context: `lor_id`, `district`, `district_id`,
  `neighborhood`, `neighborhood_id`.
- Geometry: point geometry in WGS84, kept for the unified POI table.
  

In [2]:
v0_path = Path("cache/vets_osm_berlin_with_lor_latest_v0.csv")

df = pd.read_csv(v0_path)

print("v0 shape:", df.shape)
df.head()

v0 shape: (175, 25)


,source_osm_id,name,addr:street,addr:housenumber,addr:postcode,addr:city,phone,contact:phone,email,contact:email,website,contact:website,opening_hours,operator,wheelchair,wheelchair:description,emergency,lat,lon,geometry,lor_id,district,district_id,neighborhood,neighborhood_id
0,"('node', 268917040)",Tierarztpraxis am Urban,Baerwaldstraße,69,10961.0,Berlin,NaN,NaN,NaN,NaN,NaN,NaN,"Mo-Sa 10:00-12:00, Mo 17:00-19:00, Tu,We,Fr 16...",NaN,no,NaN,NaN,52.495684,13.405233,POINT (13.4052329 52.4956842),re_ortsteil.0202,Friedrichshain-Kreuzberg,11002002,Kreuzberg,202
1,"('node', 299795048)",Dr. med. vet. Elke Hartwig,Straße 48,67,13125.0,Berlin,+49 30 9437820,NaN,NaN,NaN,http://www.tierarztpraxis-hartwig.de/,NaN,"Mo,Tu,Th,Fr 10:00-12:00, Mo-Fr 15:00-18:00",NaN,limited,NaN,NaN,52.606286,13.479555,POINT (13.4795548 52.60628629999999),re_ortsteil.0305,Pankow,11003003,Karow,305
2,"('node', 347294456)",Tierarztpraxis Dr. Bernhard Sörensen,Königsberger Straße,36,12207.0,Berlin,+49 30 7738321,NaN,NaN,NaN,https://www.tierarztpraxis-soerensen.de/,NaN,"Mo-Fr 09:00-20:00; Sa, Su 10:00-18:00",NaN,yes,NaN,NaN,52.429722,13.320133,POINT (13.3201326 52.4297216),re_ortsteil.0602,Steglitz-Zehlendorf,11006006,Lichterfelde,602
3,"('node', 394867279)",Tierarztpraxis Jeanette Koepsel,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,52.535199,13.270573,POINT (13.2705734 52.5351995),re_ortsteil.0503,Spandau,11005005,Siemensstadt,503
4,"('node', 411550894)",Kleintierarztpraxis Berlin Kaulsdorf,Planitzstraße,19,12621.0,Berlin,+49 30 53018585,NaN,info@tierarzt-kaulsdorf.de,NaN,https://www.tierarzt-kaulsdorf.de/,NaN,"Mo-Fr 09:00-19:00 open ""tel. Terminvereinbarun...",Dr. Berit Miels;Dr. Mathias Kochert,NaN,NaN,NaN,52.509511,13.589635,POINT (13.5896353 52.50951139999999),re_ortsteil.1003,Marzahn-Hellersdorf,11010010,Kaulsdorf,1003


In [3]:
df.columns.tolist()

['source_osm_id',
 'name',
 'addr:street',
 'addr:housenumber',
 'addr:postcode',
 'addr:city',
 'phone',
 'contact:phone',
 'email',
 'contact:email',
 'website',
 'contact:website',
 'opening_hours',
 'operator',
 'wheelchair',
 'wheelchair:description',
 'emergency',
 'lat',
 'lon',
 'geometry',
 'lor_id',
 'district',
 'district_id',
 'neighborhood',
 'neighborhood_id']

## 2. Helper functions

We define helper functions to:

- Build a robust `clinic_name` with sensible fallbacks.
- Compose a human-readable `full_address` while keeping structured address fields.
- Derive a simple `operating_days` label from `opening_hours` (heuristic).
- Build an aggregated `contact_info` string from structured contact fields.
- Combine wheelchair-related tags into an `accessibility_features` description.

When values are missing, we prefer to keep them as NULL (`NaN` / `pd.NA`)
rather than forcing empty strings. This makes it easier to filter in SQL later.

In [4]:
import pandas as pd


def build_clinic_name(row: pd.Series) -> object:
    """
    Derive a clinic_name with sensible fallbacks:
    1. Use OSM 'name' if available.
    2. Else use 'operator' if available.
    3. Else fall back to a generic label based on address.
    
    If nothing is available, return <NA>.
    """
    name = row.get("name")
    operator = row.get("operator")
    street = row.get("addr:street")
    housenumber = row.get("addr:housenumber")

    if isinstance(name, str) and name.strip():
        return name.strip()
    if isinstance(operator, str) and operator.strip():
        return operator.strip()

    street_s = street.strip() if isinstance(street, str) else ""
    house_s = housenumber.strip() if isinstance(housenumber, str) else ""
    if street_s or house_s:
        return f"Veterinary clinic at {street_s} {house_s}".strip()

    # If we really have no information at all:
    return pd.NA


def build_full_address(row: pd.Series) -> str:
    """
    Build a human-readable full address string from individual OSM address
    components. This is a convenience field and should not replace the
    structured address columns.
    Returns an empty string only if we truly cannot assemble any part.
    """
    street = row.get("addr:street")
    housenumber = row.get("addr:housenumber")
    postcode = row.get("addr:postcode")
    city = row.get("addr:city")

    street_s = street.strip() if isinstance(street, str) else ""
    house_s = housenumber.strip() if isinstance(housenumber, str) else ""

    # Postcode can be string or numeric
    if isinstance(postcode, str):
        postcode_s = postcode.strip()
    elif isinstance(postcode, (int, float)) and not pd.isna(postcode):
        postcode_s = str(int(postcode))
    else:
        postcode_s = ""

    city_s = city.strip() if isinstance(city, str) else ""

    parts = []
    street_part = " ".join([p for p in [street_s, house_s] if p])
    if street_part:
        parts.append(street_part)
    if postcode_s or city_s:
        parts.append(" ".join([p for p in [postcode_s, city_s] if p]))

    return ", ".join(parts)


def derive_operating_days(opening_hours) -> str:
    """
    Heuristic derivation of operating_days from an OSM 'opening_hours' string.
    This is intentionally simple and should be treated as an approximate label.
    """
    if not isinstance(opening_hours, str) or not opening_hours.strip():
        return "Unknown"

    oh = opening_hours

    # Very rough heuristics based on common patterns
    if "Mo-Su" in oh or "Mo-Sun" in oh or "MoSu" in oh.replace(" ", ""):
        return "Mon–Sun"
    if "Mo-Sa" in oh or "Mo-Sat" in oh.replace(" ", ""):
        return "Mon–Sat"
    if "Mo-Fr" in oh or "Mo-Fri" in oh.replace(" ", ""):
        return "Mon–Fri"
    if "Tu-Sa" in oh or "Tu-Sat" in oh.replace(" ", ""):
        return "Tue–Sat"

    # Fallback: detect presence of weekend days
    has_sat = "Sa" in oh or "Sat" in oh
    has_sun = "Su" in oh or "Sun" in oh

    if has_sat and has_sun:
        return "Includes weekend"
    if has_sat:
        return "Includes Saturday"
    if has_sun:
        return "Includes Sunday"

    return "Unknown"


def build_contact_info(row: pd.Series) -> str:
    """
    Build a simple aggregated contact_info field based on the 'main' phone,
    email and website. We keep structured fields separately as well.
    """
    phone = row.get("phone_main")
    email = row.get("email_main")
    website = row.get("website_main")

    parts = []
    if isinstance(phone, str) and phone.strip():
        parts.append(f"Phone: {phone.strip()}")
    if isinstance(email, str) and email.strip():
        parts.append(f"Email: {email.strip()}")
    if isinstance(website, str) and website.strip():
        parts.append(f"Web: {website.strip()}")

    return " | ".join(parts)


def build_accessibility_features(row: pd.Series) -> str:
    """
    Combine wheelchair tags into a simple accessibility_features description.
    Returns empty string if nothing is provided.
    """
    wheelchair = row.get("wheelchair")
    wheelchair_desc = row.get("wheelchair:description")

    wheelchair_s = wheelchair.strip() if isinstance(wheelchair, str) else ""
    wheelchair_desc_s = wheelchair_desc.strip() if isinstance(wheelchair_desc, str) else ""

    parts = []
    if wheelchair_s:
        parts.append(f"wheelchair={wheelchair_s}")
    if wheelchair_desc_s:
        parts.append(wheelchair_desc_s)

    return " | ".join(parts)


def has_minimal_info(row: pd.Series) -> bool:
    """
    Define a simple rule for minimal record quality for application use:
    at least one of clinic_name, full_address, phone_main, website_main
    should be present.
    """
    fields = [
        row.get("clinic_name"),
        row.get("full_address"),
        row.get("phone_main"),
        row.get("website_main"),
    ]
    for v in fields:
        if isinstance(v, str) and v.strip():
            return True
    return False

## 3. Derive core cleaned columns

We now build the core cleaned fields:

- `clinic_name` – normalized, with fallbacks.
- Structured address:
  - `addr_street`, `addr_housenumber`, `addr_postcode`, `addr_city`
  - `full_address` as a convenience string.
- Spatial context:
  - `district`, `district_id`
  - `neighborhood`, `neighborhood_id`
  - `lor_id`
- Basic service flag:
  - `services_offered` based on the `emergency` tag for now.
- Operating info:
  - `operating_hours` as the raw `opening_hours` string.
  - `operating_days` as a heuristic label.
- Contact:
  - structured fields: `phone_main`, `email_main`, `website_main`.
  - aggregated: `contact_info`.
- Accessibility:
  - `accessibility_features` from wheelchair tags.
- Location and provenance:
  - `latitude`, `longitude` from `lat` / `lon`.
  - `data_source` and `source_osm_id`.
- Geometry:
  - `geometry` column propagated from v0 for later use in the unified POI table.
- Record quality:
  - `has_minimum_info` flag to identify very sparse records.

In [5]:
df_clean = pd.DataFrame()

# 1) Clinic name with fallbacks
df_clean["clinic_name"] = df.apply(build_clinic_name, axis=1)

# 2) Structured address fields
df_clean["addr_street"] = df["addr:street"].astype("string")
df_clean["addr_housenumber"] = df["addr:housenumber"].astype("string")
df_clean["addr_postcode"] = df["addr:postcode"].astype("string")
df_clean["addr_city"] = df["addr:city"].astype("string")

# Convenience full_address
df_clean["full_address"] = df.apply(build_full_address, axis=1)

# 3) District / neighborhood context (names and IDs)
df_clean["district"] = df.get("district", pd.Series(dtype="string")).astype("string")
df_clean["district_id"] = df.get("district_id", pd.Series(dtype="string")).astype("string")
df_clean["neighborhood"] = df.get("neighborhood", pd.Series(dtype="string")).astype("string")
df_clean["neighborhood_id"] = df.get("neighborhood_id", pd.Series(dtype="string")).astype("string")
df_clean["lor_id"] = df.get("lor_id", pd.Series(dtype="string")).astype("string")

# 4) Services offered – currently derived from 'emergency' tag only
def map_services_offered(emergency_value):
    if isinstance(emergency_value, str) and emergency_value.strip():
        return "emergency"
    return ""

df_clean["services_offered"] = df.get("emergency", pd.Series(dtype="string")).apply(map_services_offered)

# 5) Operating hours & days
df_clean["operating_hours"] = df.get("opening_hours", pd.Series(dtype="string")).astype("string")
df_clean["operating_days"] = df_clean["operating_hours"].apply(derive_operating_days)

# 6) Contact: main structured fields (choose best candidate per type)
phone_candidates = [
    df.get("phone", pd.Series(dtype="string")).astype("string"),
    df.get("contact:phone", pd.Series(dtype="string")).astype("string"),
]
df_clean["phone_main"] = phone_candidates[0]
mask_empty_phone = df_clean["phone_main"].isna() | (df_clean["phone_main"].str.strip() == "")
df_clean.loc[mask_empty_phone, "phone_main"] = phone_candidates[1][mask_empty_phone]

email_candidates = [
    df.get("email", pd.Series(dtype="string")).astype("string"),
    df.get("contact:email", pd.Series(dtype="string")).astype("string"),
]
df_clean["email_main"] = email_candidates[0]
mask_empty_email = df_clean["email_main"].isna() | (df_clean["email_main"].str.strip() == "")
df_clean.loc[mask_empty_email, "email_main"] = email_candidates[1][mask_empty_email]

website_candidates = [
    df.get("website", pd.Series(dtype="string")).astype("string"),
    df.get("contact:website", pd.Series(dtype="string")).astype("string"),
]
df_clean["website_main"] = website_candidates[0]
mask_empty_website = df_clean["website_main"].isna() | (df_clean["website_main"].str.strip() == "")
df_clean.loc[mask_empty_website, "website_main"] = website_candidates[1][mask_empty_website]

# Also keep raw contact tags (optional, for debugging / future use)
raw_cols_map = {
    "phone": "phone_raw",
    "contact:phone": "contact_phone_raw",
    "email": "email_raw",
    "contact:email": "contact_email_raw",
    "website": "website_raw",
    "contact:website": "contact_website_raw",
}

for src_col, dst_col in raw_cols_map.items():
    if src_col in df.columns:
        df_clean[dst_col] = df[src_col].astype("string")

# Aggregated contact_info
df_clean["contact_info"] = df_clean.apply(build_contact_info, axis=1)

# 7) Accessibility features
df_clean["accessibility_features"] = df.apply(build_accessibility_features, axis=1)

# 8) Location and provenance
df_clean["latitude"] = df.get("lat")
df_clean["longitude"] = df.get("lon")

# Data source string – uses snapshot date for traceability
df_clean["data_source"] = (
    f"OSM amenity=veterinary, Berlin, snapshot {SNAPSHOT_DATE}, via OSMNX"
)

# Keep source_osm_id for traceability back to OSM
if "source_osm_id" in df.columns:
    df_clean["source_osm_id"] = df["source_osm_id"].astype("string")

# Geometry: keep geometry for unified POI table
if "geometry" in df.columns:
    df_clean["geometry"] = df["geometry"]

# 9) Record quality flag
df_clean["has_minimum_info"] = df_clean.apply(has_minimal_info, axis=1)

df_clean.head()

,clinic_name,addr_street,addr_housenumber,addr_postcode,addr_city,full_address,district,district_id,neighborhood,neighborhood_id,lor_id,services_offered,operating_hours,operating_days,phone_main,email_main,website_main,phone_raw,contact_phone_raw,email_raw,contact_email_raw,website_raw,contact_website_raw,contact_info,accessibility_features,latitude,longitude,data_source,source_osm_id,geometry,has_minimum_info
0,Tierarztpraxis am Urban,Baerwaldstraße,69,10961.0,Berlin,"Baerwaldstraße 69, 10961 Berlin",Friedrichshain-Kreuzberg,11002002,Kreuzberg,202,re_ortsteil.0202,,"Mo-Sa 10:00-12:00, Mo 17:00-19:00, Tu,We,Fr 16...",Mon–Sat,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,wheelchair=no,52.495684,13.405233,"OSM amenity=veterinary, Berlin, snapshot 2025-...","('node', 268917040)",POINT (13.4052329 52.4956842),True
1,Dr. med. vet. Elke Hartwig,Straße 48,67,13125.0,Berlin,"Straße 48 67, 13125 Berlin",Pankow,11003003,Karow,305,re_ortsteil.0305,,"Mo,Tu,Th,Fr 10:00-12:00, Mo-Fr 15:00-18:00",Mon–Fri,+49 30 9437820,<NA>,http://www.tierarztpraxis-hartwig.de/,+49 30 9437820,<NA>,<NA>,<NA>,http://www.tierarztpraxis-hartwig.de/,<NA>,Phone: +49 30 9437820 | Web: http://www.tierar...,wheelchair=limited,52.606286,13.479555,"OSM amenity=veterinary, Berlin, snapshot 2025-...","('node', 299795048)",POINT (13.4795548 52.60628629999999),True
2,Tierarztpraxis Dr. Bernhard Sörensen,Königsberger Straße,36,12207.0,Berlin,"Königsberger Straße 36, 12207 Berlin",Steglitz-Zehlendorf,11006006,Lichterfelde,602,re_ortsteil.0602,,"Mo-Fr 09:00-20:00; Sa, Su 10:00-18:00",Mon–Fri,+49 30 7738321,<NA>,https://www.tierarztpraxis-soerensen.de/,+49 30 7738321,<NA>,<NA>,<NA>,https://www.tierarztpraxis-soerensen.de/,<NA>,Phone: +49 30 7738321 | Web: https://www.tiera...,wheelchair=yes,52.429722,13.320133,"OSM amenity=veterinary, Berlin, snapshot 2025-...","('node', 347294456)",POINT (13.3201326 52.4297216),True
3,Tierarztpraxis Jeanette Koepsel,<NA>,<NA>,<NA>,<NA>,,Spandau,11005005,Siemensstadt,503,re_ortsteil.0503,,<NA>,Unknown,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,,52.535199,13.270573,"OSM amenity=veterinary, Berlin, snapshot 2025-...","('node', 394867279)",POINT (13.2705734 52.5351995),True
4,Kleintierarztpraxis Berlin Kaulsdorf,Planitzstraße,19,12621.0,Berlin,"Planitzstraße 19, 12621 Berlin",Marzahn-Hellersdorf,11010010,Kaulsdorf,1003,re_ortsteil.1003,,"Mo-Fr 09:00-19:00 open ""tel. Terminvereinbarun...",Mon–Fri,+49 30 53018585,info@tierarzt-kaulsdorf.de,https://www.tierarzt-kaulsdorf.de/,+49 30 53018585,<NA>,info@tierarzt-kaulsdorf.de,<NA>,https://www.tierarzt-kaulsdorf.de/,<NA>,Phone: +49 30 53018585 | Email: info@tierarzt-...,,52.509511,13.589635,"OSM amenity=veterinary, Berlin, snapshot 2025-...","('node', 411550894)",POINT (13.5896353 52.50951139999999),True


## 4. Normalize empty strings to NULL

For downstream SQL usage, it's often more convenient to have missing values
as NULL rather than empty strings. Here we convert empty strings to `NaN`
for selected columns where "empty" means "unknown".

In [6]:
cols_where_null_is_better = [
    "clinic_name",
    "addr_street",
    "addr_housenumber",
    "addr_postcode",
    "addr_city",
    "full_address",
    "district",
    "district_id",
    "neighborhood",
    "neighborhood_id",
    "services_offered",
    "operating_hours",
    "operating_days",
    "phone_main",
    "email_main",
    "website_main",
    "contact_info",
    "accessibility_features",
]

for col in cols_where_null_is_better:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].replace("", pd.NA)

df_clean.head()

,clinic_name,addr_street,addr_housenumber,addr_postcode,addr_city,full_address,district,district_id,neighborhood,neighborhood_id,lor_id,services_offered,operating_hours,operating_days,phone_main,email_main,website_main,phone_raw,contact_phone_raw,email_raw,contact_email_raw,website_raw,contact_website_raw,contact_info,accessibility_features,latitude,longitude,data_source,source_osm_id,geometry,has_minimum_info
0,Tierarztpraxis am Urban,Baerwaldstraße,69,10961.0,Berlin,"Baerwaldstraße 69, 10961 Berlin",Friedrichshain-Kreuzberg,11002002,Kreuzberg,202,re_ortsteil.0202,<NA>,"Mo-Sa 10:00-12:00, Mo 17:00-19:00, Tu,We,Fr 16...",Mon–Sat,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,wheelchair=no,52.495684,13.405233,"OSM amenity=veterinary, Berlin, snapshot 2025-...","('node', 268917040)",POINT (13.4052329 52.4956842),True
1,Dr. med. vet. Elke Hartwig,Straße 48,67,13125.0,Berlin,"Straße 48 67, 13125 Berlin",Pankow,11003003,Karow,305,re_ortsteil.0305,<NA>,"Mo,Tu,Th,Fr 10:00-12:00, Mo-Fr 15:00-18:00",Mon–Fri,+49 30 9437820,<NA>,http://www.tierarztpraxis-hartwig.de/,+49 30 9437820,<NA>,<NA>,<NA>,http://www.tierarztpraxis-hartwig.de/,<NA>,Phone: +49 30 9437820 | Web: http://www.tierar...,wheelchair=limited,52.606286,13.479555,"OSM amenity=veterinary, Berlin, snapshot 2025-...","('node', 299795048)",POINT (13.4795548 52.60628629999999),True
2,Tierarztpraxis Dr. Bernhard Sörensen,Königsberger Straße,36,12207.0,Berlin,"Königsberger Straße 36, 12207 Berlin",Steglitz-Zehlendorf,11006006,Lichterfelde,602,re_ortsteil.0602,<NA>,"Mo-Fr 09:00-20:00; Sa, Su 10:00-18:00",Mon–Fri,+49 30 7738321,<NA>,https://www.tierarztpraxis-soerensen.de/,+49 30 7738321,<NA>,<NA>,<NA>,https://www.tierarztpraxis-soerensen.de/,<NA>,Phone: +49 30 7738321 | Web: https://www.tiera...,wheelchair=yes,52.429722,13.320133,"OSM amenity=veterinary, Berlin, snapshot 2025-...","('node', 347294456)",POINT (13.3201326 52.4297216),True
3,Tierarztpraxis Jeanette Koepsel,<NA>,<NA>,<NA>,<NA>,<NA>,Spandau,11005005,Siemensstadt,503,re_ortsteil.0503,<NA>,<NA>,Unknown,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,52.535199,13.270573,"OSM amenity=veterinary, Berlin, snapshot 2025-...","('node', 394867279)",POINT (13.2705734 52.5351995),True
4,Kleintierarztpraxis Berlin Kaulsdorf,Planitzstraße,19,12621.0,Berlin,"Planitzstraße 19, 12621 Berlin",Marzahn-Hellersdorf,11010010,Kaulsdorf,1003,re_ortsteil.1003,<NA>,"Mo-Fr 09:00-19:00 open ""tel. Terminvereinbarun...",Mon–Fri,+49 30 53018585,info@tierarzt-kaulsdorf.de,https://www.tierarzt-kaulsdorf.de/,+49 30 53018585,<NA>,info@tierarzt-kaulsdorf.de,<NA>,https://www.tierarzt-kaulsdorf.de/,<NA>,Phone: +49 30 53018585 | Email: info@tierarzt-...,<NA>,52.509511,13.589635,"OSM amenity=veterinary, Berlin, snapshot 2025-...","('node', 411550894)",POINT (13.5896353 52.50951139999999),True


## 5. Quick QA checks

We perform some basic checks:

- Null counts per column.
- How many records have missing `clinic_name`, `district`, `neighborhood`, or coordinates.
- Distribution of `has_minimum_info`.

In [7]:
print("Null counts per column:")
print(df_clean.isna().sum())

print("\nEmpty or NULL clinic_name rows:", df_clean["clinic_name"].isna().sum())
print("NULL district rows:", df_clean["district"].isna().sum())
print("NULL neighborhood rows:", df_clean["neighborhood"].isna().sum())
print("NULL latitude rows:", df_clean["latitude"].isna().sum())
print("NULL longitude rows:", df_clean["longitude"].isna().sum())

print("\nRecord quality (has_minimum_info value counts):")
print(df_clean["has_minimum_info"].value_counts(dropna=False))

Null counts per column:
clinic_name                 4
addr_street                48
addr_housenumber           48
addr_postcode              60
addr_city                  62
full_address               48
district                    0
district_id                 0
neighborhood                0
neighborhood_id             0
lor_id                      0
services_offered          172
operating_hours            50
operating_days              0
phone_main                 78
email_main                145
website_main               66
phone_raw                 103
contact_phone_raw         150
email_raw                 157
contact_email_raw         163
website_raw                89
contact_website_raw       151
contact_info               53
accessibility_features     93
latitude                    0
longitude                   0
data_source                 0
source_osm_id               0
geometry                    0
has_minimum_info            0
dtype: int64

Empty or NULL clinic_name rows: 

In [8]:
df_clean.columns

Index(['clinic_name', 'addr_street', 'addr_housenumber', 'addr_postcode',
       'addr_city', 'full_address', 'district', 'district_id', 'neighborhood',
       'neighborhood_id', 'lor_id', 'services_offered', 'operating_hours',
       'operating_days', 'phone_main', 'email_main', 'website_main',
       'phone_raw', 'contact_phone_raw', 'email_raw', 'contact_email_raw',
       'website_raw', 'contact_website_raw', 'contact_info',
       'accessibility_features', 'latitude', 'longitude', 'data_source',
       'source_osm_id', 'geometry', 'has_minimum_info'],
      dtype='object')

In [9]:
# Inspect low-information clinics
low_info = df_clean[df_clean["has_minimum_info"] == False].copy()
cols_view = [
    "clinic_name",
    "full_address",
    "district",
    "neighborhood",
    "phone_main",
    "email_main",
    "website_main",
    "latitude",
    "longitude",
]
low_info[cols_view]

,clinic_name,full_address,district,neighborhood,phone_main,email_main,website_main,latitude,longitude
20,<NA>,<NA>,Charlottenburg-Wilmersdorf,Schmargendorf,<NA>,<NA>,<NA>,52.478761,13.282550
40,<NA>,<NA>,Spandau,Spandau,<NA>,<NA>,<NA>,52.547729,13.202257
107,<NA>,<NA>,Pankow,Wilhelmsruh,<NA>,<NA>,<NA>,52.587788,13.365155
141,<NA>,<NA>,Marzahn-Hellersdorf,Mahlsdorf,<NA>,<NA>,<NA>,52.487317,13.604680


## 6. Export cleaned v1 dataset

We export the cleaned and normalized vet clinics dataset as **v1** under:

- `cache/vet_clinics_berlin_clean_latest_v1.csv`

This file is intended to be the main input for:

- DB loading scripts / ETL into the final `vet_clinics_table`.
- Frontend or analytics workflows that need a normalized vet clinics layer.

The final DB schema (with numeric `id`, FK `district_id`, and
`geometry` in `POINT(lon lat)` format) will be handled in a separate step.

In [10]:
output_v1_csv = Path("cache/vet_clinics_berlin_clean_latest_v1.csv")

output_v1_csv.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(output_v1_csv, index=False)

print("Exported cleaned v1 file to:")
print(" -", output_v1_csv)
print("Shape:", df_clean.shape)

Exported cleaned v1 file to:
 - cache/vet_clinics_berlin_clean_latest_v1.csv
Shape: (175, 31)
